In [1]:
# FRIDAY — ONE-CELL GOOGLE COLAB BACKEND
# Required Colab Secrets: NGROK_AUTHTOKEN, FRIDAY_API_KEY
# Optional Colab Secret: HF_TOKEN (for gated models such as Meta Llama)

import sys
import subprocess

# 1. Install what the runtime needs (reusable after Colab resets).
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.46,<5", "accelerate>=1.0",
        "fastapi", "uvicorn", "pyngrok",
    ],
    check=True,
)

import gc
import socket
import threading
import time
from contextlib import suppress

import torch
from fastapi import FastAPI, Header, HTTPException
from fastapi.concurrency import run_in_threadpool
from pydantic import BaseModel, Field
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer
from google.colab import userdata

# 2. MODEL CONFIG — only this ID needs changing for a compatible instruct model.
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 256

# 3. PERSONA — edit here whenever you want to evolve Friday's personality.
FRIDAY_SYSTEM_PROMPT = """You are Friday, a personal AI assistant.
You are helpful, practical, friendly, reliable, and calm.
Automatically respond in the language the user uses. If they use Hinglish,
respond in natural Hinglish. If they use English or Hindi, follow that language.
Give clear, useful answers. Be concise by default, but explain technical topics
step by step when helpful. For code, prioritize correctness and runnable examples.
If you don't know something, admit uncertainty instead of inventing facts.
Do not pretend to have tools, access, memory, or capabilities you don't have.
""".strip()

# 4. SECRETS (enable notebook access for each secret in Colab's key icon).
def get_secret(name, *, required=False):
    try:
        value = userdata.get(name)
        if value and value.strip():
            return value.strip()
    except Exception:
        pass
    if required:
        raise RuntimeError(
            f"Missing Colab Secret: {name}. Add it in the Secrets sidebar "
            "and enable notebook access."
        )
    return None

NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN", required=True)
API_KEY = get_secret("FRIDAY_API_KEY", required=True)
HF_TOKEN = get_secret("HF_TOKEN")  # optional

if not torch.cuda.is_available():
    raise RuntimeError("GPU not available. Select Runtime > Change runtime type > T4 GPU.")

# If this cell is re-run in the same runtime, close our previous tunnel/server.
if "_friday_server" in globals():
    _friday_server.should_exit = True
    _previous_thread = globals().get("_friday_thread")
    if _previous_thread and _previous_thread.is_alive():
        _previous_thread.join(timeout=10)
    _previous_tunnel = globals().get("_friday_tunnel")
    if _previous_tunnel:
        with suppress(Exception):
            ngrok.disconnect(_previous_tunnel.public_url)
    for _name in ("_friday_model", "_friday_tokenizer"):
        if _name in globals():
            del globals()[_name]
    gc.collect()
    torch.cuda.empty_cache()

# 5. LOAD MODEL
print("Loading:", MODEL_ID)
_friday_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, token=HF_TOKEN
)
_friday_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=torch.float16,  # T4-compatible; may need changes for other GPUs/models
    device_map="auto",
)
_friday_model.eval()

# Single generation at a time avoids overlapping GPU requests.
_generation_lock = threading.Lock()

def generate_response(message: str) -> str:
    messages = [
        {"role": "system", "content": FRIDAY_SYSTEM_PROMPT},
        {"role": "user", "content": message},
    ]
    prompt = _friday_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = _friday_tokenizer(prompt, return_tensors="pt").to(
        _friday_model.device
    )
    with _generation_lock, torch.inference_mode():
        output_ids = _friday_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=(
                _friday_tokenizer.pad_token_id
                if _friday_tokenizer.pad_token_id is not None
                else _friday_tokenizer.eos_token_id
            ),
        )
    reply_tokens = output_ids[0, inputs["input_ids"].shape[-1]:]
    return _friday_tokenizer.decode(reply_tokens, skip_special_tokens=True).strip()

# 6. API — matches the existing Friday Next.js /api/chat route exactly.
app = FastAPI(title="Friday Colab Backend")

class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=4000)

@app.get("/health")
def health():
    return {"status": "running", "model": MODEL_ID}

@app.post("/chat")
async def chat(data: ChatRequest, x_api_key: str = Header()):
    import secrets
    if not secrets.compare_digest(x_api_key, API_KEY):
        raise HTTPException(status_code=401, detail="Unauthorized")
    message = data.message.strip()
    if not message:
        raise HTTPException(status_code=422, detail="Empty message")
    result = await run_in_threadpool(generate_response, message)
    return {"response": result}

# 7. UVICORN — select a free port to avoid colliding with earlier Colab cells.
with socket.socket() as _sock:
    _sock.bind(("127.0.0.1", 0))
    PORT = _sock.getsockname()[1]

_friday_server = __import__("uvicorn").Server(
    __import__("uvicorn").Config(
        app, host="127.0.0.1", port=PORT, log_level="warning"
    )
)
_friday_thread = threading.Thread(
    target=_friday_server.run, daemon=True
)
_friday_thread.start()

for _ in range(100):
    if _friday_server.started:
        break
    if not _friday_thread.is_alive():
        raise RuntimeError("FastAPI failed to start. Check runtime logs.")
    time.sleep(0.1)
else:
    raise RuntimeError("FastAPI startup timed out.")

# 8. EXPOSE THROUGH NGROK
ngrok.set_auth_token(NGROK_AUTHTOKEN)
_friday_tunnel = ngrok.connect(addr=PORT, proto="http")

print("\nFRIDAY IS READY")
print("Model:", MODEL_ID)
print("Ngrok URL:", _friday_tunnel.public_url)
print("Health:", _friday_tunnel.public_url + "/health")
print("API key: loaded securely from Colab Secrets (not printed).")
print("Keep Colab connected; update NGROK_BASE_URL if this URL changes.")


Loading: Qwen/Qwen2.5-1.5B-Instruct


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


FRIDAY IS READY
Model: Qwen/Qwen2.5-1.5B-Instruct
Ngrok URL: https://preconceptional-roland-nondisposable.ngrok-free.dev
Health: https://preconceptional-roland-nondisposable.ngrok-free.dev/health
API key: loaded securely from Colab Secrets (not printed).
Keep Colab connected; update NGROK_BASE_URL if this URL changes.
